# 05_01_Alinhamento_e_Retornos

Este notebook executa a **Etapa V** do pipeline de tratamento de dados: o alinhamento temporal de cotações diárias (ações, Ibovespa, CDI e SELIC) por interseção cronológica estrita, seguido da conversão geométrica de preços para o domínio de retornos simples e logarítmicos.

**SRP (Single Responsibility Principle):** Este script possui apenas a responsabilidade de indexação temporal comum e mapeamento de retornos brutos.

## 1. Configurações, Parâmetros e Resolução de Caminhos

In [1]:
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
from utils.config_loader import carregar_parametros
from utils.alinhamento import obter_calendario_comum, alinhar_painel
from utils.conversoes import calcular_retornos_simples, calcular_retornos_log

warnings.filterwarnings("ignore")

# Carregar parâmetros centrais do config.json
config = carregar_parametros("config.json")
EXCLUIR_PRECO_CORROMPIDO = config["EXCLUIR_PRECO_CORROMPIDO"]
LIMIAR_PRECO_MAX = config["LIMIAR_PRECO_MAX"]

# Resolução de caminhos a partir da raiz do projeto
parent_path = Path.cwd().parent.parent
INPUT_DIR_MATRIZ_PRECOS = parent_path / "data" / "Matriz_Precos"
INPUT_DIR_IBOV = parent_path / "data" / "Ibov" / "Ibov_Diario"
INPUT_DIR_CDI = parent_path / "data" / "CDI"
INPUT_DIR_SELIC = parent_path / "data" / "Selic"
OUTPUT_DIR = parent_path / "data" / "Tickers"
DIR_PAINEL = parent_path / "data" / "Painel_Dados"
DIR_RETORNOS = parent_path / "data" / "Retornos"

for d in (DIR_PAINEL, DIR_RETORNOS, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
print(f"[OK] Toggle de exclusão por preço corrompido: {EXCLUIR_PRECO_CORROMPIDO}")
print(f"[OK] Limiar de preço máximo: R$ {LIMIAR_PRECO_MAX:,.2f}")

[OK] Toggle de exclusão por preço corrompido: False
[OK] Limiar de preço máximo: R$ 1,000.00


## 2. Carregamento dos Insumos de Produção

In [2]:
def _carregar_csv(path, date_col):
    if not path.exists():
        raise FileNotFoundError(f"{path.name} não encontrado em {path.parent}.")
    # Suporta separadores ',' e ';' automaticamente
    df = pd.read_csv(path, sep=None, engine='python')
    if date_col not in df.columns:
        raise ValueError(f"Coluna de data '{date_col}' não encontrada em {path.name}.")
    df[date_col] = pd.to_datetime(df[date_col])
    return df.set_index(date_col).sort_index()

# Carrega a base de preços dos 118 ativos
precos = _carregar_csv(INPUT_DIR_MATRIZ_PRECOS / "Matriz_precos_sanitizada.csv", "Data")
print(f"Matriz de preços: {precos.shape[1]} ativos × {precos.shape[0]} pregões "
      f"({precos.index.min().date()} → {precos.index.max().date()})")

# Carrega benchmarks de mercado e taxas livres de risco
ibov_close = _carregar_csv(INPUT_DIR_IBOV / "ibov_diario_2010_2026.csv", "data")["ibov_close"].rename("IBOV_close")
cdi_diario = _carregar_csv(INPUT_DIR_CDI / "cdi_diario_bcb_2010_atual.csv", "data")["cdi_diario"].rename("CDI_diario")
selic_diario = _carregar_csv(INPUT_DIR_SELIC / "selic_diario_bcb_2010_atual.csv", "data")["selic_diario"].rename("SELIC_diario")

print(f"IBOVESPA: {len(ibov_close):,} | CDI: {len(cdi_diario):,} | SELIC: {len(selic_diario):,} obs.")

# Registro histórico de exclusões prévias
_excl = INPUT_DIR_MATRIZ_PRECOS / "tickers_excluidos.csv"
excluidos_prev = (pd.read_csv(_excl) if _excl.exists()
                  else pd.DataFrame(columns=["ticker","presenca_pct","primeiro_pregao_real","motivo"]))
print(f"Registro de exclusões anterior: {len(excluidos_prev)} tickers")

Matriz de preços: 102 ativos × 3967 pregões (2010-01-04 → 2025-12-30)
IBOVESPA: 3,967 | CDI: 4,019 | SELIC: 4,019 obs.
Registro de exclusões anterior: 378 tickers


## 3. Diagnóstico e Filtro Opcional de Preços Máximos (Etapa VI)

In [3]:
rets_diag = precos.pct_change()
integridade = pd.DataFrame({
    "preco_inicial": precos.iloc[0], "preco_final": precos.iloc[-1],
    "preco_max": precos.max(), "preco_min": precos.min(),
    "amplitude": precos.max() / precos.min(),
    "saltos_>50pct": (rets_diag.abs() > 0.50).sum(),
})
mask_corrompido = integridade["preco_max"] > LIMIAR_PRECO_MAX
print(f"=== Diagnóstico: preço máximo > R$ {LIMIAR_PRECO_MAX:,.0f} ({int(mask_corrompido.sum())}) ===")
print(integridade[mask_corrompido].sort_values("preco_max", ascending=False)
      [["preco_inicial","preco_max","amplitude","saltos_>50pct"]].to_string())

if EXCLUIR_PRECO_CORROMPIDO:
    precos_VI = precos.drop(columns=integridade.index[mask_corrompido])
    novos = pd.DataFrame({"ticker": integridade.index[mask_corrompido], "presenca_pct": np.nan,
                          "primeiro_pregao_real": np.nan,
                          "motivo": f"preco max > R$ {LIMIAR_PRECO_MAX:.0f} (Etapa VI)"})
    excluidos_prev = pd.concat([excluidos_prev, novos], ignore_index=True)
    print(f"\n[Etapa VI LIGADA] Excluídos {int(mask_corrompido.sum())} ativos. Universo: {precos.shape[1]} → {precos_VI.shape[1]} ativos.")
else:
    precos_VI = precos
    print(f"\n[Etapa VI DESLIGADA] Mantidos todos os {precos_VI.shape[1]} ativos.")

=== Diagnóstico: preço máximo > R$ 1,000 (1) ===
       preco_inicial     preco_max    amplitude  saltos_>50pct
GFSA3  12,025.947461 12,927.532550 2,756.403529              1

[Etapa VI DESLIGADA] Mantidos todos os 102 ativos.


## 4. Alinhamento Temporal por Interseção Cronológica

In [4]:
calendario = obter_calendario_comum([precos_VI, ibov_close, cdi_diario, selic_diario])
print(f"Calendário de interseção: {len(calendario):,} pregões "
      f"({calendario.min().date()} → {calendario.max().date()})")

# Auditoria de espaçamento cronológico (feriados)
gaps = pd.Series(calendario).diff().dt.days
print("\nMaiores intervalos entre pregões (feriados prolongados esperados):")
print(pd.DataFrame({"data": calendario, "gap_dias": gaps.values}).nlargest(5,"gap_dias").to_string(index=False))

Calendário de interseção: 3,967 pregões (2010-01-04 → 2025-12-30)

Maiores intervalos entre pregões (feriados prolongados esperados):
      data  gap_dias
2010-02-17  5.000000
2011-03-09  5.000000
2011-04-25  5.000000
2012-02-22  5.000000
2012-12-26  5.000000


## 5. Consolidação e Salvamento do Painel de Preços Alinhados

In [5]:
painel = alinhar_painel(precos_VI, ibov_close, cdi_diario, selic_diario, calendario)
print(f"Painel Consolidado: {painel.shape[0]:,} pregões × {painel.shape[1]} colunas")

# Exportação dos preços de fechamento sincronizados
painel.to_parquet(DIR_PAINEL / "painel_alinhado.parquet", engine="pyarrow")
painel.to_csv(DIR_PAINEL / "painel_alinhado.csv", date_format="%Y-%m-%d", float_format="%.8f")
print("[OK] Painel de preços alinhados salvo com sucesso em CSV e Parquet.")

Painel Consolidado: 3,967 pregões × 105 colunas


[OK] Painel de preços alinhados salvo com sucesso em CSV e Parquet.


## 6. Geração de Matrizes de Retornos Diários

In [6]:
cols_acoes = [c for c in painel.columns if c.startswith("ACAO_")]
precos_acoes = painel[cols_acoes]

# Computação vetorial de retornos simples e contínuos
ret_simples = calcular_retornos_simples(precos_acoes)
ret_log = calcular_retornos_log(precos_acoes)

# Retornos do Benchmark (Ibovespa)
ibov_ret_simples = calcular_retornos_simples(painel["IBOV_close"])
ibov_ret_log = calcular_retornos_log(painel["IBOV_close"])

# Reindexação de Taxas Livres de Risco
rf = painel[["CDI_diario","SELIC_diario"]].reindex(ret_simples.index)

print(f"Matriz de retornos discretos: {ret_simples.shape[0]} pregões × {ret_simples.shape[1]} ativos")

# Validação geométrica estrita das matrizes
assert np.allclose(ret_log.values, np.log1p(ret_simples.values), atol=1e-10)
print("[OK] Identidade log ≡ ln(1+simples) validada em todas as células.")

Matriz de retornos discretos: 3966 pregões × 102 ativos
[OK] Identidade log ≡ ln(1+simples) validada em todas as células.


## 7. Exportação de Artefatos de Retorno e Tickers

In [7]:
def _salvar(df, nome, ddir=DIR_RETORNOS):
    df.to_csv(ddir / f'{nome}.csv', index=True, date_format='%Y-%m-%d', float_format='%0.8f')
    df.to_parquet(ddir / f'{nome}.parquet', engine='pyarrow')
    print(f'  • {nome}.csv e {nome}.parquet gerados.')

print('[+] Gravando retornos brutos em:', DIR_RETORNOS, '\n')
_salvar(ret_simples, 'retornos_simples')
_salvar(ret_log, 'retornos_log')
_salvar(pd.DataFrame({'ibov_ret_simples': ibov_ret_simples, 'ibov_ret_log': ibov_ret_log}), 'ibov_retornos')
_salvar(rf.rename(columns=str.lower), 'rf_diario')

# Atualização da lista de ativos
pd.Series([c.replace('ACAO_','') for c in cols_acoes], name='ticker').to_csv(OUTPUT_DIR / 'tickers_finais.csv', index=False)
excluidos_prev.to_csv(OUTPUT_DIR / 'tickers_excluidos.csv', index=False)

print('\n[OK] Finalizado notebook de Alinhamento e Conversão de Retornos.')

[+] Gravando retornos brutos em: C:\VSCodeWorkspace\1_TCC_Final\data\Retornos 



  • retornos_simples.csv e retornos_simples.parquet gerados.


  • retornos_log.csv e retornos_log.parquet gerados.
  • ibov_retornos.csv e ibov_retornos.parquet gerados.
  • rf_diario.csv e rf_diario.parquet gerados.

[OK] Finalizado notebook de Alinhamento e Conversão de Retornos.


# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

> Rule A, Birmingham A, Zuniga C, Altintas I, Huang S-C, Knight R, Moshiri N, Nguyen MH,
> Rosenthal SB, Pérez F, Rose PW. *Ten simple rules for writing and sharing computational analyses
> in Jupyter Notebooks.* PLOS Comput Biol. 2019;15(7):e1007007.

| Regra | Tema | Status | Evidência / Aplicação no NB 05_01 (Alinhamento de Retornos) |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | ✅ | Narrativa estruturada com introdução metodológica, células de cálculo comentadas e interpretação dos outputs. |
| 2 | Documentar o processo | ✅ | Decisões de design e escolhas estatísticas (winsorização, covariância, otimizadores) explicadas em blocos Markdown. |
| 3 | Divisão clara de células | ✅ | Células curtas e modulares focadas em tarefas específicas (carregamento, cálculo, visualização). |
| 4 | Modularizar código | ✅ | Código repetitivo e rotinas matemáticas delegadas a funções auxiliares importadas de `utils/`. |
| 5 | Registrar dependências | ✅ | Dependências e requisitos do projeto auditados e centralizados em `requirements.txt` e `requirements.py`. |
| 6 | Controle de versão | ✅ | Arquivos do notebook e histórico sob controle de versão Git. |
| 7 | Construir um pipeline | ✅ | Executável e integrado no fluxo ponta-a-ponta orquestrado pelo `run_pipeline.py`. |
| 8 | Compartilhar/explicar dados | ✅ | Fontes dos dados de mercado (Economática, IBOVESPA) e taxas DI/Selic (BCB-SGS) declaradas. |
| 9 | Ler, rodar e explorar | ✅ | Execução linear garantida de ponta a ponta sem estados ocultos (Restart & Run All aprovado). |
| 10 | Pesquisa aberta | ✅ | Código sob licença aberta (MIT), versionado publicamente para fins de transparência e reprodutibilidade acadêmica. |
